In [1]:
import os
import re
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split

print("TensorFlow version:", tf.__version__)


TensorFlow version: 2.19.0


In [2]:
from google.colab import drive
import zipfile

drive.mount("/content/drive")

zip_path = "/content/drive/MyDrive/Parkinson's Dataset Final.zip"
extract_path = "/content/dataset"

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_path)

BASE = "/content/dataset/Parkinson's Dataset Final"
print("Dataset extracted to:", BASE)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset extracted to: /content/dataset/Parkinson's Dataset Final


In [3]:
records = []

def scan_images(root_dir):
    for root, _, files in os.walk(root_dir):
        for f in files:
            if f.lower().endswith((".jpg", ".png", ".jpeg")):
                full_path = os.path.join(root, f)

                if "healthy" in full_path.lower():
                    label = 0
                elif "parkinson" in full_path.lower():
                    label = 1
                else:
                    continue

                records.append({
                    "path": full_path,
                    "label": label
                })

# New Hand PD
scan_images(os.path.join(BASE, "New hand pd Dataset"))
# Old Hand PD
scan_images(os.path.join(BASE, "Hand Pd"))

image_df = pd.DataFrame(records)

print("Total images:", len(image_df))
print("\nClass distribution:")
print(image_df["label"].value_counts())


Total images: 1320

Class distribution:
label
1    871
0    449
Name: count, dtype: int64


In [4]:
train_df, test_df = train_test_split(
    image_df,
    test_size=0.2,
    stratify=image_df["label"],
    random_state=42
)

print("Train:", train_df["label"].value_counts())
print("Test:", test_df["label"].value_counts())


Train: label
1    697
0    359
Name: count, dtype: int64
Test: label
1    174
0     90
Name: count, dtype: int64


In [5]:
import cv2

IMG_SIZE = 224

def load_image(path):
    img = cv2.imread(path.decode(), cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img.astype(np.float32) / 255.0
    return img[..., np.newaxis]


In [6]:
train_df, test_df = train_test_split(
    image_df,
    test_size=0.2,
    stratify=image_df["label"],
    random_state=42
)


In [7]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["label"]),
    y=train_df["label"]
)

class_weights = {
    0: class_weights[0],
    1: class_weights[1]
}

class_weights


{0: np.float64(1.4707520891364902), 1: np.float64(0.757532281205165)}

In [8]:
def load_image(path):
    img = cv2.imread(path.decode(), cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (224,224))
    img = img.astype(np.float32) / 255.0
    return img[..., np.newaxis]


In [9]:
def make_dataset(df, batch_size=16):
    paths = df["path"].values
    labels = df["label"].values

    def _load(path, label):
        img = tf.numpy_function(load_image, [path], tf.float32)
        img.set_shape((224,224,1))
        return img, label

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(_load, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds


In [10]:
class PatchEmbedding(layers.Layer):
    def __init__(self, patch_size=16, embed_dim=526):
        super().__init__()
        self.proj = layers.Conv2D(
            filters=embed_dim,
            kernel_size=patch_size,
            strides=patch_size,
            padding="valid"
        )

    def call(self, x):
        x = self.proj(x)
        x = tf.reshape(x, (tf.shape(x)[0], -1, x.shape[-1]))
        return x


In [11]:
class PositionalEncoding(layers.Layer):
    def __init__(self, num_patches, dim):
        super().__init__()
        self.embedding = layers.Embedding(num_patches, dim)

    def call(self, x):
        positions = tf.range(tf.shape(x)[1])
        return x + self.embedding(positions)


In [15]:
class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads=4):
        super().__init__()

        self.attn = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim
        )

        self.norm1 = layers.LayerNormalization()
        self.norm2 = layers.LayerNormalization()

        self.mlp = tf.keras.Sequential([
            layers.Dense(embed_dim * 4, activation="gelu"),
            layers.Dense(embed_dim)
        ])

    def call(self, x):
        x = self.norm1(x + self.attn(x, x))
        x = self.norm2(x + self.mlp(x))
        return x


In [16]:
def projection_stage(x, out_dim):
    x = layers.LayerNormalization()(x)
    x = layers.Dense(out_dim)(x)
    return x


In [17]:
def build_custom_vit_feature_extractor():
    inputs = tf.keras.Input(shape=(224,224,1))

    # Patch embedding
    x = PatchEmbedding(patch_size=16, embed_dim=526)(inputs)
    x = PositionalEncoding(num_patches=196, embed_dim=526)(x)

    # Stage 1 — 526
    x = TransformerBlock(526)(x)

    # Stage 2 — 256
    x = projection_stage(x, 256)
    x = TransformerBlock(256)(x)

    # Stage 3 — 128
    x = projection_stage(x, 128)
    x = TransformerBlock(128)(x)

    # Stage 4 — 64
    x = projection_stage(x, 64)
    x = TransformerBlock(64)(x)

    # Stage 5 — 32
    x = projection_stage(x, 32)
    x = TransformerBlock(32)(x)

    # Stage 6 — 16
    x = projection_stage(x, 16)
    x = TransformerBlock(16)(x)

    # Stage 7 — 8
    x = projection_stage(x, 8)
    x = TransformerBlock(8)(x)

    # Stage 8 — 4
    x = projection_stage(x, 4)
    x = TransformerBlock(4)(x)

    # Feature vector
    x = layers.GlobalAveragePooling1D()(x)

    return tf.keras.Model(inputs, x, name="Custom_ViT_Feature_Extractor")


In [18]:
vit_extractor = build_custom_vit_feature_extractor()
vit_extractor.summary()


TypeError: PositionalEncoding.__init__() got an unexpected keyword argument 'embed_dim'